# Extract Neighbor Features
Uploads: `config.py`, `models.py`, `features.py`, `batches.py`, `extract_features.py`  
Set `COLX_DIR` below to wherever you placed those files in Drive.

In [ ]:
!pip install -q datasets transformers torch pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

# ── Edit this path ────────────────────────────────────────────────────────────
COLX_DIR = Path('/content/drive/MyDrive/Neural_and_Neighbors/src')
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(COLX_DIR))
assert COLX_DIR.exists(), f'Directory not found: {COLX_DIR}'

In [ ]:
import random
import torch
from models import load_model_and_processor, load_base_model_and_processor, load_data
from batches import process_dataset_in_batches

random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('\nLoading fine-tuned model...')
processor, model = load_model_and_processor(device)

print('\nLoading base model...')
base_processor, base_model = load_base_model_and_processor(device)

In [ ]:
print('Loading dataset...')
splits = load_data()

split_map = {
    'train': splits['train'],
    'val':   splits['validation'],
    'test':  splits['test'],
}

caption_pool = splits['train']['text'].str.split('\n', n=1).str[1].tolist()
print(f'Caption pool size: {len(caption_pool)}')

In [ ]:
for split_name in ['train', 'val', 'test']:
    df = split_map[split_name]
    has_labels = split_name != 'test'

    print(f"\n{'='*60}")
    print(f'Extracting: {split_name}  ({len(df)} samples)')
    print(f"{'='*60}")

    features_df = process_dataset_in_batches(
        model, processor, df,
        has_labels=has_labels,
        base_model=base_model,
        base_processor=base_processor,
        caption_pool=caption_pool,
        n_neighbors=15,
    )

    out_path = COLX_DIR / f'{split_name}_neighbor_only.csv'
    features_df.to_csv(out_path, index=False)
    print(f'Saved → {out_path}  ({len(features_df)} rows, {len(features_df.columns)} cols)')